# Path-dependent Python strategies

Some strategies cannot be written as a table of order intent. A rule that
enters only once the previous position is confirmed closed, or that waits
thirty seconds after a fill before acting, depends on what already happened
rather than on the current row.

That is what path-dependent means, and it needs the strategy to carry state
between events. The cost is a crossing into Python for every event the
strategy sees, which is why this recipe closes by naming the cheaper
boundaries and when to prefer them.

Signals and command tables are the fastest strategy boundary because the
replay stays entirely in Rust. Some strategies genuinely need state,
fill-driven decisions, or timers. This recipe uses the opt-in Python callback
surface without giving strategy code direct access to borrowed engine
internals.

## Terms used here

| term              | meaning |
| ----------------- | --- |
| callback          | your Python function, called by the engine when something happens |
| path-dependent    | a decision that depends on what already happened, not only on current data |
| stateful          | the strategy carries variables between events |
| timer             | a callback scheduled for a future instant rather than triggered by data |
| strategy identity | a stable name for a strategy, so its runs stay comparable across reruns |
| determinism       | the same inputs producing the same outputs, every time, which reruns verify |

New to any of these? [GLOSSARY.md](../../GLOSSARY.md) defines them at more
length, along with every other term the cookbook uses.

In [1]:
import h5i_db
from h5i_db import backtest
import cookbook_utils as cu

INSTRUMENT_ID = "RATE-CUT-YES"
MARKET_CUT = "callback-market-cut"
SECOND = 1_000_000_000

fixture = cu.make_backtest_fixture(steps=150, instrument_id=INSTRUMENT_ID)
db = h5i_db.Database(cu.fresh_db("07_python_strategy_callbacks"), create=True)
for name, table in fixture.items():
    db.create_table(name, table.schema, time_column="ts_init")
    db.append(name, table, note="deterministic callback fixture")
db.snapshot(
    MARKET_CUT,
    tables=["instruments", "book_deltas", "trades"],
    note="Approved market-data cut for callback examples",
)

{'name': 'callback-market-cut',
 'created_at_ns': 1785438032640251650,
 'note': 'Approved market-data cut for callback examples',
 'entries': {'79a409b9-2295-47bb-a949-cb1e6e230df5': {'table_name': 'instruments',
   'sequence': 1,
   'manifest_checksum': 'fa06c5d44262dd4da1d7c8dfca0bd46506dd783f7198b484ba4db1be88b32bb8'},
  '830b4c39-dac0-43c7-93da-f06dc9a11d47': {'table_name': 'book_deltas',
   'sequence': 1,
   'manifest_checksum': '9b229223071d52600d7b6832ed61debb41ae0c0710407f0aedefc62d72e95d17'},
  'a24830cb-e4a5-4ab5-abc8-56ee7db8fbda': {'table_name': 'trades',
   'sequence': 1,
   'manifest_checksum': 'c0021c1d76eca3f019b71381b5017953c986d19d4e22546732e15f5485c7f295'}},
 'checksum': '9c0f16f326cca917c7aedddde8b442b8c1dae0b0278f279ae70c64eda8896f46'}

## Write a state machine with explicit effects

Callback inputs are ordinary dictionaries. A callback returns `None`, one
action mapping, or an iterable of mappings. The engine applies those actions
after the callback, preserving causal event ordering. This strategy:

1. schedules an entry timer from the first observable market event;
2. submits a market buy when that timer fires;
3. schedules an exit only after the entry fill is confirmed; and
4. submits a reduce-only sell from the exit timer.

In [2]:
class TimedRoundTrip(backtest.EventStrategy):
    def __init__(self):
        self.entry_scheduled = False
        self.exit_scheduled = False
        self.fill_log = []

    def on_event(self, context, event):
        assert context["now"] == event["ts_init"]
        if not self.entry_scheduled:
            self.entry_scheduled = True
            return {
                "action": "timer",
                "name": "enter",
                "ts": context["now"] + 20 * SECOND,
            }
        return None

    def on_timer(self, context, event):
        if event["name"] == "enter":
            return {
                "action": "submit",
                "client_order_id": "entry",
                "instrument_id": INSTRUMENT_ID,
                "side": "buy",
                "quantity": 25.0,
                "tag": "timed-entry",
            }
        if event["name"] == "exit":
            return {
                "action": "submit",
                "client_order_id": "exit",
                "instrument_id": INSTRUMENT_ID,
                "side": "sell",
                "quantity": 25.0,
                "reduce_only": True,
                "tag": "fill-driven-exit",
            }
        raise ValueError(f"unexpected timer {event['name']!r}")

    def on_fill(self, context, event):
        self.fill_log.append(event)
        if event["tag"] == "timed-entry" and not self.exit_scheduled:
            self.exit_scheduled = True
            return {
                "action": "timer",
                "name": "exit",
                "ts": event["ts"] + 60 * SECOND,
            }
        return None


strategy = TimedRoundTrip()

`strategy_id` is persisted with the run. In packaged research code,
`run_strategy` can derive it from class source; an explicit version is often
preferable in notebooks and production because code review can tie it to a
release or commit.

In [3]:
result = backtest.run_strategy(
    db,
    "timed-round-trip",
    strategy,
    strategy_id="cookbook.TimedRoundTrip:v1",
    starting_cash=10_000.0,
    data=backtest.DataConfig(
        snapshot=MARKET_CUT,
        minimum_coverage=0.95,
    ),
    execution=backtest.ExecutionConfig(
        fee_kind="prediction_market",
        fee_rate=0.02,
        latency_nanos=1_000_000,
    ),
    risk=backtest.RiskConfig(
        max_order_quantity=25.0,
        max_abs_position=25.0,
        max_open_orders=2,
    ),
    output=backtest.OutputConfig(equity_interval_nanos=5 * SECOND),
    metadata={"purpose": "callback and timer contract demonstration"},
)
result

{'run_id': 'timed-round-trip',
 'fork': 'bt-timed-round-trip',
 'digest': 'eca719a0093562462aae3e40a900136ca5d999425bf0b842306322f1d4c11589',
 'starting_cash': 10000.0,
 'final_cash': 9999.427559665,
 'realized_pnl': -0.572440335,
 'commissions': 0.249940335,
 'funding_paid': 0.0,
 'fills': 2,
 'orders': 2,
 'records_processed': 300,
 'simulated_through_ns': 1780322550000000000,
 'equity_points': 31,
 'settlement_applied': False,
 'coverage': None,
 'liquidations': 0,
 'rejected_for_margin': 0,
 'self_trades_prevented': 0,
 'metrics': {'orders_submitted': 2,
  'orders_filled': 2,
  'orders_cancelled_unfilled': 0,
  'orders_rejected_margin': 0,
  'orders_rejected_risk': 0,
  'orders_rejected_self_trade': 0,
  'fills_taker': 2,
  'fills_maker': 0,
  'book_gaps': 0,
  'liquidations': 0},
 'warnings': [],
 'cached': False,
 'trial_digest': '49fb5ae60bfd8a456a04423cf6f2e1e2f5f6240bf54b0f0ebb9a5983dd6b4218'}

Fill callbacks ran on the same strategy object under the GIL. The persisted
tables remain the source of truth; local state is useful for decisions and
diagnostics, not as the audit record.

In [4]:
orders = result.orders.to_pandas()
fills = result.fills.to_pandas()
assert result["fills"] == 2
assert [fill["tag"] for fill in strategy.fill_log] == [
    "timed-entry",
    "fill-driven-exit",
]
assert fills["tag"].tolist() == ["timed-entry", "fill-driven-exit"]
assert result.positions.to_pandas()["quantity"].abs().sum() < 1e-9
orders[["order_id", "side", "quantity", "filled", "status", "tag"]]

,order_id,side,quantity,filled,status,tag
0,1,buy,25.0,25.0,filled,timed-entry
1,2,sell,25.0,25.0,filled,fill-driven-exit


Callback runs are reproducible when the same strategy implementation is
supplied. Verification creates an isolated rerun, compares metrics and all
authoritative output tables, then removes the temporary fork.

In [5]:
verification = result.verify(strategy=TimedRoundTrip())
assert verification["verified"]
verification

{'left': 'timed-round-trip',
 'right': 'verify-timed-round-trip-074dfedf982d',
 'same_digest': False,
 'metrics': {'records_processed': {'left': 300,
   'right': 300,
   'delta': 0,
   'equal': True},
  'orders': {'left': 2, 'right': 2, 'delta': 0, 'equal': True},
  'fills': {'left': 2, 'right': 2, 'delta': 0, 'equal': True},
  'final_cash': {'left': 9999.427559665,
   'right': 9999.427559665,
   'delta': 0.0,
   'equal': True},
  'realized_pnl': {'left': -0.572440335,
   'right': -0.572440335,
   'delta': 0.0,
   'equal': True},
  'commissions': {'left': 0.249940335,
   'right': 0.249940335,
   'delta': 0.0,
   'equal': True},
  'equity_points': {'left': 31, 'right': 31, 'delta': 0, 'equal': True},
  'coverage': {'left': None, 'right': None, 'delta': None, 'equal': True}},
 'tables_equal': {'bt_orders': True,
  'bt_fills': True,
  'bt_positions': True,
  'bt_equity': True},
 'verified': True}

## When to choose each strategy boundary

| boundary | best for | performance | lifecycle |
|---|---|---|---|
| signals | vectorized entries/exits and target positions | native hot loop | submit |
| commands | quoting and predetermined execution schedules | native hot loop | submit/amend/cancel |
| Python callbacks | state machines, fill reactions, timers | one GIL crossing per callback | full |

Prefer the simplest boundary that expresses the strategy. Callback
flexibility is valuable, but it should be an explicit choice rather than an
accidental tax on every backtest.

In [6]:
db.close()